# Week 1 · 手写 Multi-Head Attention

> **学习模式**:核心算法由你自己写。我只给骨架、维度提示和验证用的参考实现。

## 本周目标

1. **手写 Single-Head Attention** → 理解 Q/K/V 在做什么(内容寻址)
2. **手写 Multi-Head Attention** → 理解 `MHA(x) = Concat(heads)·W_O` 等价形式
3. **证明等价性**:multi-head 拼接 = 把 `W_Q, W_K, W_V` 切成 h 份分别算再拼
4. **Causal masking**:为 decoder-style self-attention 加上下三角 mask
5. **跟 `nn.MultiheadAttention` 对数值**:确认你的实现正确

## Deliverable

- `mha_scratch.py`(从本 notebook 抽出,关掉参考再写一遍)
- 1 页数学推导笔记(维度流、等价性证明)写在 `../notes/week1_reviewer.md`

## 审稿人自问清单

- 为什么除以 `sqrt(d_k)`?不除会怎样?
- Softmax 为什么沿 key 维度?沿 query 维度会发生什么?
- Multi-head 跟 single-head with larger d 的区别是什么?(表达力 vs. 参数量)
- Causal mask 应该在 softmax 前还是后加?为什么?

## 0. 环境与工具

In [ ]:
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = "mps" if torch.backends.mps.is_available() else "cpu"
torch.manual_seed(42)
print("device:", device, "| torch:", torch.__version__)

## 1. Single-Head Attention(你来写)

**输入**:`x` of shape `(B, T, d_model)`

**要做的事**:
1. 用三个 Linear 投影到 `Q, K, V`,每个形状 `(B, T, d_k)`
2. 计算 `scores = Q @ K^T / sqrt(d_k)`,形状 `(B, T, T)`
3. (可选)加 causal mask
4. `attn = softmax(scores, dim=?)` ← 想清楚沿哪一维
5. `out = attn @ V`,形状 `(B, T, d_k)`

**提示**:
- `torch.tril(torch.ones(T, T))` 可以造下三角矩阵
- Mask 的典型做法:`scores.masked_fill(mask == 0, float('-inf'))` **在 softmax 前**
- 初始化用 `nn.Linear(d_model, d_k, bias=False)`(bias 先不加,减少变量)

In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, d_model: int, d_k: int, causal: bool = True):
        super().__init__()
        # TODO: 定义 W_q, W_k, W_v(三个 Linear)
        # TODO: 保存 d_k, causal
        raise NotImplementedError("你的实现")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model)
        # TODO:
        # 1. 投影 Q K V
        # 2. scores = Q @ K^T / sqrt(d_k)
        # 3. causal mask(如果 self.causal)
        # 4. softmax → attn weights
        # 5. out = attn @ V
        # return out, attn  (把 attn 也返回,方便可视化)
        raise NotImplementedError("你的实现")

In [ ]:
# 小测试:前向能跑通 + 形状对
B, T, d_model, d_k = 2, 5, 16, 8
x = torch.randn(B, T, d_model)
sha = SingleHeadAttention(d_model, d_k, causal=True)
out, attn = sha(x)
assert out.shape == (B, T, d_k), f"got {out.shape}"
assert attn.shape == (B, T, T)
# causal 检查:attn 应该是下三角
assert torch.allclose(attn.triu(1), torch.zeros_like(attn), atol=1e-6), "causal mask 没生效"
# 每行概率和为 1
assert torch.allclose(attn.sum(-1), torch.ones(B, T), atol=1e-5)
print("✅ single-head 基本检查通过")

## 2. Multi-Head Attention(你来写)

**核心等价性**:`h` 个独立 head 拼接 == 把 `W_Q` 看成一个大矩阵,输出 reshape 成 `(B, T, h, d_k)` 再换维成 `(B, h, T, d_k)` 算 attention,最后拼回去。

**要做的事**:
1. 一次投影得到 Q/K/V,形状 `(B, T, d_model)`,其中 `d_model = h * d_k`
2. Reshape + transpose → `(B, h, T, d_k)`
3. 每个 head 独立算 attention(其实就是 batched matmul)
4. 拼接:`(B, h, T, d_k)` → `(B, T, h*d_k)`
5. 输出投影 `W_O`:`(B, T, d_model)` → `(B, T, d_model)`

**关键 API**:
- `.view()` / `.reshape()` 改形状
- `.transpose(1, 2)` 交换 head 和 seq 维
- `torch.matmul` 支持 broadcasting,`(B, h, T, d_k) @ (B, h, d_k, T)` 直接就是每个 head 的 scores

**审稿人问题**(想完再写):
- 为什么 reshape + transpose 后就能用一次 matmul 算所有 head?
- `.view()` 和 `.reshape()` 的区别在哪?什么时候 `.view()` 会报错?

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, causal: bool = True):
        super().__init__()
        assert d_model % num_heads == 0
        # TODO: 保存 d_model, num_heads, d_k = d_model // num_heads, causal
        # TODO: 定义 W_q, W_k, W_v, W_o(都是 d_model → d_model)
        raise NotImplementedError("你的实现")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, d_model)
        B, T, _ = x.shape
        # TODO:
        # 1. 投影 Q K V:(B, T, d_model)
        # 2. Reshape 成 (B, T, h, d_k) 再 transpose 到 (B, h, T, d_k)
        # 3. scores = Q @ K.transpose(-2, -1) / sqrt(d_k)   → (B, h, T, T)
        # 4. causal mask
        # 5. softmax → attn
        # 6. out = attn @ V   → (B, h, T, d_k)
        # 7. transpose + reshape 回 (B, T, d_model)
        # 8. 过 W_o
        # return out, attn
        raise NotImplementedError("你的实现")

In [ ]:
# 形状测试
B, T, d_model, h = 2, 7, 32, 4
x = torch.randn(B, T, d_model)
mha = MultiHeadAttention(d_model, h, causal=True)
out, attn = mha(x)
assert out.shape == (B, T, d_model)
assert attn.shape == (B, h, T, T)
print("✅ MHA 形状对")

## 3. 跟 `nn.MultiheadAttention` 对数值
## 3. Numerical Comparison With `nn.MultiheadAttention`

**目的**:确认你的实现数值上等价(允许小误差)。

**坑**:PyTorch 的 `nn.MultiheadAttention` 默认 `batch_first=False`,要传 `batch_first=True`。权重也要你手动从你的模块同步过去(或者反过来)。

这一节留给你自己搭 —— 核心技能:**读 PyTorch 文档,把两边的权重形状对齐**。如果卡住了,来问我。

**Goal**: verify that your implementation is numerically equivalent up to small floating-point differences.

**Gotcha**: PyTorch's `nn.MultiheadAttention` defaults to `batch_first=False`, so you should pass `batch_first=True`. You also need to manually copy weights from your module into PyTorch's module, or the other way around.

This section is intentionally left for you to wire up yourself. The key skill here is **reading PyTorch docs and aligning parameter shapes correctly across two implementations**. If you get stuck, ask.

In [ ]:
# TODO: 构造 torch 的 nn.MultiheadAttention,把你的 W_q/W_k/W_v/W_o 权重复制过去,
#       同一输入喂进去,验证 output 数值差 < 1e-5

## 4. 等价性证明(写在 notes/week1_reviewer.md)

**命题**:用 `h` 个独立的 head,每个 head 有自己的 `W_q^{(i)}, W_k^{(i)}, W_v^{(i)} ∈ R^{d_model × d_k}`,拼接输出再过 `W_o`,等价于 reshape 版本。

写出来的过程应该包括:

1. 给出两种形式的数学定义
2. 证明 block-diagonal 的 `W_q` 在「投影 + reshape + 分 head 算」上的行为与「每个 head 独立投影」是一样的
3. 讨论参数量:`4 · d_model²`(Q/K/V/O)和 single-head with d=d_model 的差别在哪?

**为什么这步重要**:这是你理解 MHA 为什么「便宜又有效」的根。后面看 Flash Attention、MQA、GQA 都会回到这个等价式。

## 5. 下一步

- [ ] 本 notebook 核心 cell 全填完,测试通过
- [ ] 关掉 notebook,开 `mha_scratch.py` 从空白重写一遍
- [ ] 写 `../notes/week1_reviewer.md`(等价性证明 + 审稿人自问清单的答案)
- [ ] 精读《Attention Is All You Need》,用你自己的实现对着论文图 2 过一遍
- [ ] 读 Phuong & Hutter《Formal Algorithms for Transformers》